In [1]:
# =============================================================================
# Data Processing
# =============================================================================
import pandas as pd

# =============================================================================
# Text Processing
# =============================================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
# =============================================================================
# Load the preprocessed dataset
# =============================================================================
dataset = pd.read_csv("preprocessed_dataset.csv")
print(dataset.shape)

dataset.head()

(3997391, 13)


,QuestionID,QuestionType,Category,AskerID,QuestionTime,QuestionText,AnswerText,AnswererID,AnswerTime,question_length,answer_length,semantic_text,lexical_text
0,C1Q1,yes/no,Automotive,A365S8H55GGXPD,2013-07-19,will they fit 2013 f350 dually,"It's all custom mounting, where there's a will...",AQZ8QLPPYA359,2014-12-29,6,31,Question: will they fit 2013 f350 dually\n\nAn...,fit 2013 f350 dually custom mounting theres th...
1,C1Q1,yes/no,Automotive,A365S8H55GGXPD,2013-07-19,will they fit 2013 f350 dually,You will need to drill another hole in Mud fla...,A246IDL7UXVCQO,2014-12-29,6,60,Question: will they fit 2013 f350 dually\n\nAn...,fit 2013 f350 dually need drill another hole m...
2,C1Q1,yes/no,Automotive,A365S8H55GGXPD,2013-07-19,will they fit 2013 f350 dually,"It's been a while since I installed them, but ...",A3BWPG98KF0TAV,2013-07-19,6,17,Question: will they fit 2013 f350 dually\n\nAn...,fit 2013 f350 dually since installed dont thin...
3,C1Q1,yes/no,Automotive,A365S8H55GGXPD,2013-07-19,will they fit 2013 f350 dually,1 pair rear flaps and mounting hardware.,1,2013-07-19,6,7,Question: will they fit 2013 f350 dually\n\nAn...,fit 2013 f350 dually 1 pair rear flaps mountin...
4,C1Q1,yes/no,Automotive,A365S8H55GGXPD,2013-07-19,will they fit 2013 f350 dually,I didn't buy these for myself I bought them fo...,A1MGZTOLD2C0VS,2013-07-19,6,24,Question: will they fit 2013 f350 dually\n\nAn...,fit 2013 f350 dually didnt buy bought soninlaw...


In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# Semantic Chunking
# Step 1: Define a sliding-window chunking function
# ──────────────────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_size=120, overlap=30):
    """
    Split a document into overlapping word-based chunks.

    Parameters
    ----------
    text : str
        Document text.

    chunk_size : int
        Maximum number of words per chunk.

    overlap : int
        Number of overlapping words between consecutive chunks.
    """

    words = str(text).split()

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunks.append(
            " ".join(words[start:end]) )

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks

In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# Semantic Chunk DataFrame
# Step 2: Split semantic documents while preserving metadata
# ──────────────────────────────────────────────────────────────────────────────
semantic_chunks = []

for _, row in dataset.iterrows():

    chunks = chunk_text(row["semantic_text"])

    for chunk_index, chunk in enumerate(chunks):

        semantic_chunks.append({

            # Unique chunk identifier
            "chunk_id":
                f"{row['QuestionID']}_chunk_{chunk_index}",

            # Parent document identifier
            "QuestionID":
                row["QuestionID"],

            # Metadata
            "Category":
                row["Category"],

            "QuestionType":
                row["QuestionType"],

            "QuestionTime":
                row["QuestionTime"],

            # Chunk order
            "chunk_index":
                chunk_index,

            # Chunk content
            "chunk_text":
                chunk,

            # Text used for embedding generation
            "search_text":
                (
                    f"Category: {row['Category']} "
                    f"Question Type: {row['QuestionType']} "
                    f"{chunk}"
                )

        }) 

semantic_chunks_df = pd.DataFrame(semantic_chunks)

print("Total Semantic Chunks:", len(semantic_chunks_df))

semantic_chunks_df.head()

Total Semantic Chunks: 4534002


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
1,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
2,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
3,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
4,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...


In [7]:
# ──────────────────────────────────────────────────────────────────────────────
# Lexical Chunk DataFrame
# Step 3: Split lexical documents while preserving metadata
# ──────────────────────────────────────────────────────────────────────────────
lexical_chunks = []

for _, row in dataset.iterrows():

    chunks = chunk_text(row["lexical_text"])

    for chunk_index, chunk in enumerate(chunks):

        lexical_chunks.append({

            # Unique chunk identifier
            "chunk_id":
                f"{row['QuestionID']}_chunk_{chunk_index}",

            # Parent document identifier
            "QuestionID":
                row["QuestionID"],

            # Metadata
            "Category":
                row["Category"],

            "QuestionType":
                row["QuestionType"],

            "QuestionTime":
                row["QuestionTime"],

            # Chunk order
            "chunk_index":
                chunk_index,

            # Chunk content
            "chunk_text":
                chunk,

            # Text used for BM25 indexing
            "search_text":
                (
                    f"{row['Category']} "
                    f"{row['QuestionType']} "
                    f"{chunk}"
                )

        })

lexical_chunks_df = pd.DataFrame(lexical_chunks)

print("Total Lexical Chunks:", len(lexical_chunks_df))

lexical_chunks_df.head()

Total Lexical Chunks: 4193484


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,fit 2013 f350 dually custom mounting theres th...,Automotive yes/no fit 2013 f350 dually custom ...
1,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,fit 2013 f350 dually need drill another hole m...,Automotive yes/no fit 2013 f350 dually need dr...
2,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,fit 2013 f350 dually since installed dont thin...,Automotive yes/no fit 2013 f350 dually since i...
3,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,fit 2013 f350 dually 1 pair rear flaps mountin...,Automotive yes/no fit 2013 f350 dually 1 pair ...
4,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,fit 2013 f350 dually didnt buy bought soninlaw...,Automotive yes/no fit 2013 f350 dually didnt b...


In [8]:
# =============================================================================
# Inspect generated chunks
# =============================================================================

print("Semantic Chunk Shape :", semantic_chunks_df.shape)
print("Lexical Chunk Shape  :", lexical_chunks_df.shape)

semantic_chunks_df.head(3)

Semantic Chunk Shape : (4534002, 8)
Lexical Chunk Shape  : (4193484, 8)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text
0,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
1,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...
2,C1Q1_chunk_0,C1Q1,Automotive,yes/no,2013-07-19,0,Question: will they fit 2013 f350 dually Answe...,Category: Automotive Question Type: yes/no Que...


In [9]:
# =============================================================================
# Display one complete chunk
# =============================================================================

sample = 0

print("Chunk ID:")
print(semantic_chunks_df.loc[sample, "chunk_id"])

print("\nMetadata")
print("-------------------------")
print("QuestionID :", semantic_chunks_df.loc[sample, "QuestionID"])
print("Category   :", semantic_chunks_df.loc[sample, "Category"])
print("Type       :", semantic_chunks_df.loc[sample, "QuestionType"])
print("Time       :", semantic_chunks_df.loc[sample, "QuestionTime"])

print("\nChunk Text")
print("-------------------------")
print(semantic_chunks_df.loc[sample, "chunk_text"])

print("\nSearch Text")
print("-------------------------")
print(semantic_chunks_df.loc[sample, "search_text"])

Chunk ID:
C1Q1_chunk_0

Metadata
-------------------------
QuestionID : C1Q1
Category   : Automotive
Type       : yes/no
Time       : 2013-07-19

Chunk Text
-------------------------
Question: will they fit 2013 f350 dually Answer: it's all custom mounting, where there's a will there's a way, don't see why not, figure 15minutes per mud flap, there isn't any hardware included just the stamped sheet metal pieces

Search Text
-------------------------
Category: Automotive Question Type: yes/no Question: will they fit 2013 f350 dually Answer: it's all custom mounting, where there's a will there's a way, don't see why not, figure 15minutes per mud flap, there isn't any hardware included just the stamped sheet metal pieces


In [ ]:
# =============================================================================
# Save the chunks dataset
# =============================================================================
dataset.to_csv("chunks.dataset.csv", index=False)

print("chunks dataset saved successfully.")